# Use this to generate data

In [ ]:
import numpy as np
import math
import pandas as pd
from pathlib import Path
from joblib import Parallel, delayed
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

PAPER_DIR = Path(r"E:\PAPER")

code_dir = PAPER_DIR / "code"
data_dir = PAPER_DIR / "data"
fig_dir = PAPER_DIR / "figures"
paper_dir = PAPER_DIR / "paper"

data_generation_code_dir = code_dir / "data_generation"
plotting_code_dir = code_dir / "plotting"

kerr_data_dir = data_dir / "kerr"
kerr_data_error_analysis_dir = data_dir / "error_analysis"
kerr_critical_curves_data_dir = kerr_data_dir / "critical_curves"
kerr_critical_quantities_data_dir = kerr_data_dir / "critical_quantities"
kerr_trajectories_data_dir = kerr_data_dir / "trajectories"
kerr_critical_trajectories_data_dir = kerr_data_dir / "critical_trajectories"
kerr_critical_curves_data_dir_ivp = kerr_critical_curves_data_dir / "IVP"

quasi_kerr_data_dir = data_dir / "quasi_kerr"
quasi_kerr_critical_curves_data_dir = quasi_kerr_data_dir / "critical_curves"
quasi_kerr_critical_quantities_data_dir = quasi_kerr_data_dir / "critical_quantities"
quasi_kerr_trajectories_data_dir = quasi_kerr_data_dir / "trajectories"
quasi_kerr_critical_trajectories_data_dir = quasi_kerr_data_dir / "critical_trajectories"
qk_testing_data_dir = quasi_kerr_data_dir / "testing"

kerr_fig_dir = fig_dir / "kerr"
quasi_kerr_fig_dir = fig_dir / "quasi_kerr"

paper_drafts_dir = paper_dir / "drafts"
paper_figure_dir = paper_dir / "final_figures"

In [105]:
# ============================================================
# BLACK HOLE / NUMERICAL PARAMETERS
# ============================================================

M = 1.0
r_obs = 40.0
phi_obs = 0.0

stepSize = 0.01
max_steps = 100000

r_max_cut = 50.0

H_tol = 1e-6
bisection_tol = 1e-8

# ============================================================
# DEFAULT B SCAN
# ============================================================

n_rays = 20
b_values = np.linspace(0, 10, n_rays)


# ============================================================
# TRACKED TRAJECTORY QUANTITIES
# ============================================================

TRACK_KEYS = [
    "t", "r", "th", "phi",
    "pt", "pr", "pth", "pphi",
    "t_dot", "r_dot", "th_dot", "phi_dot",
    "x", "y", "z"
]

In [107]:
# ============================================================
# Kerr metric (Boyer-Lindquist)
# ============================================================
def metric_cov_kerr(x, a):
    t, r, th, ph = x
    cos_th, sin_th = math.cos(th), math.sin(th)
    Sigma = r**2 + (a * cos_th)**2
    Delta = r**2 - 2 * M * r + a**2
    g = np.zeros((4, 4))
    g[0, 0] = -(1 - 2 * M * r / Sigma)        #g_{tt}
    g[0, 3] = g[3, 0] = -2 * M * r * a * sin_th**2 / Sigma    #g_{t,phi} = g_{phi,t}
    g[1, 1] = Sigma / Delta    #g_{rr}
    g[2, 2] = Sigma      #g_{theta,theta}
    g[3, 3] = (r**2 + a**2 + 2 * M * r * a**2 * sin_th**2 / Sigma) * sin_th**2  #g_{phi,phi}
    return g

In [109]:
# ============================================================
# Quasi-Kerr metric (Boyer-Lindquist) 
# ============================================================
def h_contra_quasikerr(x, a):
    t, r, th, ph = x
    cos_th, sin_th = math.cos(th), math.sin(th)
    Sigma = r**2 + (a * cos_th)**2
    Delta = r**2 - 2 * M * r + a**2
    if r <= 2 * M:
        raise ValueError("Quasi-Kerr perturbation requires r > 2M.")
    ln_term = math.log(r / (r - 2 * M))

    F1 = (-5 * (r - M) * (2 * M**2 + 6 * M * r - 3 * r**2)/ (8 * M * r * (r - 2 * M)) - 15 * r * (r - 2 * M)/ (16 * M**2)* ln_term)
    F2 = (5 * (2 * M**2 - 3 * M * r - 3 * r**2)/ (8 * M * r) + 15 * (r**2 - 2 * M**2)/ (16 * M**2)* ln_term)

    angular_term = 1 - 3 * cos_th**2
    
    h_contra = np.zeros((4, 4))
    h_contra[0, 0] = (1 - 2 * M / r)**(-1) * angular_term * F1      #h^{tt}
    h_contra[1, 1] = (1 - 2 * M / r) * angular_term * F1   #h^{rr}
    h_contra[2, 2] = -(1 / r**2) * angular_term * F2     #h^{theta,theta}
    h_contra[3, 3] = -(1 / (r**2 * sin_th**2)) * angular_term * F2  #h^{phi,phi}
    return h_contra

In [111]:
from scipy.optimize import brentq

def qk_cutoff_function(r, a, epsilon_qk, fraction=0.5):

    x = np.array([0.0, r, np.pi/2, 0.0])

    g_cov = metric_cov_kerr(x, a)
    g_contra_kerr = np.linalg.inv(g_cov)

    h_contra = h_contra_quasikerr(x, a)

    return (abs(epsilon_qk * h_contra[1, 1]) - fraction * abs(g_contra_kerr[1, 1]))


def find_qk_radial_cutoff(a, epsilon_qk, fraction=0.5):

    return brentq(qk_cutoff_function, 2.0 * M + 1e-5, 10.0 * M, args=(a, epsilon_qk, fraction))

In [113]:
def metric_contra_quasikerr(x, a, epsilon_qk):

    g_cov_kerr = metric_cov_kerr(x, a)
    g_contra_kerr = np.linalg.inv(g_cov_kerr)

    h_contra = h_contra_quasikerr(x, a)

    g_contra_qk = (g_contra_kerr - epsilon_qk * h_contra)

    return g_contra_qk

def compute_contravariant_and_derivatives(x, a, metric, epsilon_qk=0.0, eps=1e-6):

    x = np.array(x, dtype=float)

    # ========================================================
    # Kerr
    # ========================================================

    if metric == "kerr":

        g_cov = metric_cov_kerr(x, a)                  # g_{mu,nu}
        
        g_contra = np.linalg.solve(g_cov, np.eye(4))   # g^{mu,nu}

        dginv = np.zeros((4, 4, 4))

        for mu in range(4):

            dx = np.zeros(4)
            dx[mu] = eps

            g_plus = metric_cov_kerr(x + dx, a)

            g_minus = metric_cov_kerr(x - dx, a)

            dg_cov_mu = (g_plus - g_minus) / (2 * eps)

            dginv[mu] = (-g_contra @ dg_cov_mu @ g_contra)


    # ========================================================
    # Quasi-Kerr
    # Metric is defined contravariantly
    # ========================================================

    elif metric == "quasi_kerr":

        if epsilon_qk == 0.0:   #Kerr
            
            g_cov = metric_cov_kerr(x, a)                  # g_{mu,nu}
        
            g_contra = np.linalg.solve(g_cov, np.eye(4))   # g^{mu,nu}
    
            dginv = np.zeros((4, 4, 4))
    
            for mu in range(4):
    
                dx = np.zeros(4)
                dx[mu] = eps
    
                g_plus = metric_cov_kerr(x + dx, a)
    
                g_minus = metric_cov_kerr(x - dx, a)
    
                dg_cov_mu = (g_plus - g_minus) / (2 * eps)
    
                dginv[mu] = (-g_contra @ dg_cov_mu @ g_contra)
        else:
                

            g_contra = metric_contra_quasikerr(x, a, epsilon_qk=epsilon_qk)
    
            dginv = np.zeros((4, 4, 4))
    
            for mu in range(4):
    
                dx = np.zeros(4)
                dx[mu] = eps
    
                g_plus = metric_contra_quasikerr(x + dx, a, epsilon_qk=epsilon_qk)
    
                g_minus = metric_contra_quasikerr(x - dx, a, epsilon_qk=epsilon_qk)
    
                dginv[mu] = (g_plus - g_minus) / (2 * eps)
    
    else:

        raise ValueError("metric must be 'kerr' or 'quasi_kerr'")


    return g_contra, dginv

# ============================================================
# Equations of Motion and Hamiltonian evolution
# ============================================================

def dstate_dlambda(state, a, metric, epsilon_qk=0.0):  
    x, p = state[:4], state[4:]
    g_contra, dginv = compute_contravariant_and_derivatives(x, a, metric=metric, epsilon_qk=epsilon_qk)
    dxdl = g_contra @ p
    dpdl = np.array([-0.5 * (p @ (dginv[mu] @ p)) for mu in range(4)])
    return np.concatenate([dxdl, dpdl])

def hamiltonian_value(p_cov, x, a, metric, epsilon_qk=0.0):
    g_contra, _ = compute_contravariant_and_derivatives(x, a, metric=metric, epsilon_qk=epsilon_qk)
    return 0.5 * (p_cov @ (g_contra @ p_cov))

In [115]:
def shoot_ray_ivp(b, a, metric, ptheta0, theta_obs, phi_init=0.0, direction=+1, r_min_cut=None, epsilon_qk=0.0, method="DOP853", rtol=1e-8, atol=1e-10):

    x0 = np.array([0.0, r_obs, theta_obs, phi_init], dtype=float)
    pt0 = -1.0
    pphi0 = direction * abs(b)

    # --------------------------------------------------
    # Radial cutoff for Kerr and QK
    # --------------------------------------------------

    if r_min_cut is None:

        if metric == "kerr":

            r_min_cut = 1.001 * (M + math.sqrt(M**2 - a**2))

        elif metric == "quasi_kerr":

            if epsilon_qk == 0.0:

                r_min_cut = 1.001 * (M + math.sqrt(M**2 - a**2))
            else:

                r_min_cut = find_qk_radial_cutoff(a, epsilon_qk, fraction=0.5)
                print("r_min_cut:", r_min_cut)

        else:

            raise ValueError("metric must be 'kerr' or 'quasi_kerr'")
            
    # Solve null constraint H=0 for inward p_r.
    
    g_contra, _ = compute_contravariant_and_derivatives(x0, a, metric=metric, epsilon_qk=epsilon_qk)
    
    A = g_contra[1, 1]
    B = 2 * (
        g_contra[0, 1] * pt0
        + g_contra[1, 3] * pphi0
        + g_contra[2, 1] * ptheta0)
    C = (
        g_contra[0, 0] * pt0**2
        + g_contra[2, 2] * ptheta0**2
        + g_contra[3, 3] * pphi0**2
        + 2 * g_contra[0, 3] * pt0 * pphi0
        + 2 * g_contra[0, 2] * pt0 * ptheta0
        + 2 * g_contra[2, 3] * ptheta0 * pphi0)

    discriminant = B**2 - 4 * A * C
    if discriminant < 0:
        raise ValueError(
            f"Cannot find real p_r0 for b={b}, ptheta0={ptheta0}, "
            f"a={a}, theta_obs={np.rad2deg(theta_obs):.2f} deg")

    pr0 = (-B - np.sqrt(discriminant)) / (2 * A)

    state = np.concatenate([x0, [pt0, pr0, ptheta0, pphi0]])   #Initial state vector
 
    def geodesic_equations(lam, state):

        return dstate_dlambda(state, a=a, metric=metric, epsilon_qk=epsilon_qk)
        
    def plunge_event(lam, state):   

        r = state[1]

        return r - r_min_cut       #when this is 0, we are at the event horizon and stop integration


    plunge_event.terminal = True   #if this event happens then stop integration
    plunge_event.direction = -1    #only stop it if we are decreasing closer to the event horizon
    
    def escape_event(lam, state):

        r = state[1]

        return r - r_max_cut       #when this is 0, we are at "infinity" and stop integration


    escape_event.terminal = True    #if this event happens then mark it as true
    escape_event.direction = +1     #only stop it if we are increasing outward from the event horizon


    sol = solve_ivp(
        geodesic_equations,                  #equations of motion
        t_span=(0.0, max_steps * stepSize),  #maximum allowed lambda range
        y0=state,                            #initial conditions
        method="DOP853",                     #adaptive 8th order RK
        rtol=1e-8,
        atol=1e-10,
        events=[plunge_event, escape_event]  #event 0 and event 1, when these return zero an event is triggered (ie r = r_min_cut)
    )
    t_track    = sol.y[0]
    r_track    = sol.y[1]
    th_track   = sol.y[2]
    phi_track  = sol.y[3]
    
    pt_track   = sol.y[4]
    pr_track   = sol.y[5]
    pth_track  = sol.y[6]
    pphi_track = sol.y[7]
    tracks = {k: [] for k in TRACK_KEYS}
    H_values = []

    for j in range(sol.y.shape[1]):         #Loop through each step and track coordinates/fate
    
        state_j = sol.y[:, j]
    
        t, r, th, phi = state_j[:4]
        pt, pr, pth, pphi = state_j[4:]
    
        H_j = hamiltonian_value(state_j[4:], state_j[:4], a=a, metric=metric, epsilon_qk=epsilon_qk) #Diagnostic to check null constraint in each step
    
        H_values.append(H_j)
        g_contra, _ = compute_contravariant_and_derivatives(state_j[:4], a, metric=metric, epsilon_qk=epsilon_qk)
        x_dot = g_contra @ state_j[4:]
        tracks["t"].append(t)
        tracks["r"].append(r)
        tracks["th"].append(th)
        tracks["phi"].append(phi)
    
        tracks["pt"].append(pt)
        tracks["pr"].append(pr)
        tracks["pth"].append(pth)
        tracks["pphi"].append(pphi)
    
        tracks["t_dot"].append(x_dot[0])
        tracks["r_dot"].append(x_dot[1])
        tracks["th_dot"].append(x_dot[2])
        tracks["phi_dot"].append(x_dot[3])
        #Cartesian Coordinates
        tracks["x"].append(r * np.sin(th) * np.cos(phi))
        tracks["y"].append(r * np.sin(th) * np.sin(phi))
        tracks["z"].append(r * np.cos(th))
    
    H_values = np.asarray(H_values)
    abs_H = np.abs(H_values)
    
    if len(sol.t_events[0]) > 0:   #if the photon plunged, record its fate

        fate = "plunge"

    elif len(sol.t_events[1]) > 0:  #if the photon plunged, record its fate

        fate = "escape"

    else:

        fate = "unknown"
    
    phi_track = sol.y[3]

    phi_total = 0.0

    for j in range(1, len(phi_track)):   #Track total azimuthal accumulation for each trajectory
    
        phi = phi_track[j]
        phi_prev = phi_track[j - 1]
    
        dphi = (phi - phi_prev + np.pi) % (2*np.pi) - np.pi
    
        phi_total += abs(dphi)

    for key in tracks:
        tracks[key] = np.asarray(tracks[key])

    return tracks, fate, phi_total

In [119]:
#Save the same data with the same column ordering

def make_ray_record(
    tracks,
    a,
    theta_obs,
    b,
    b_unsigned,
    ptheta,
    direction,
    fate,
    phi_total,
    alpha=None,
    beta=None,
    psi=None,
    rho=None):

    if alpha is None:
        alpha = -b / np.sin(theta_obs)

    if beta is None:
        beta = ptheta

    return {
        "a": a,
        "theta_obs": theta_obs,
        "theta_obs_deg": np.rad2deg(theta_obs),

        "b": b,
        "b_unsigned": b_unsigned,
        "ptheta": ptheta,
        "direction": direction,

        "alpha": alpha,
        "beta": beta,

        "psi": psi if psi is not None else np.nan,
        "psi_deg": (
            np.rad2deg(psi)
            if psi is not None
            else np.nan
        ),

        "rho": rho if rho is not None else np.nan,

        "fate": fate,
        "phi_total": phi_total,

        **tracks
    }

In [121]:
def find_bcrit_at_ptheta_ivp(ptheta0, b_values, a, metric, theta_obs, direction=1, epsilon_qk=0.0):  #Fix ptheta (beta) and find corresponding critical impact parameter (~ alpha)

    b_scan_ray_data = []
    scan_results = []

    # --------------------------------------------------
    # Scan b values to find plunge/escape bracket
    # --------------------------------------------------

    for b in b_values:

        b_unsigned = abs(b)

        tracks, fate, phi_total = shoot_ray_ivp(b_unsigned, a, metric=metric, ptheta0=ptheta0, theta_obs=theta_obs, direction=direction, epsilon_qk=epsilon_qk)

        b_signed = direction * b_unsigned
        alpha = -b_signed / np.sin(theta_obs)
        beta = ptheta0

        print(f"b={b_signed:.4f}", f"ptheta={ptheta0:.4f}", f"alpha={alpha:.4f}", f"beta={beta:.4f}", fate)

        b_scan_ray_data.append(make_ray_record(
            tracks=tracks,
            a=a,
            theta_obs=theta_obs,
            b=b_signed,
            b_unsigned=b_unsigned,
            ptheta=ptheta0,
            direction=direction,
            fate=fate,
            phi_total=phi_total))
        
        scan_results.append((b_unsigned, fate))

    transition_brackets = []

    for i in range(len(scan_results) - 1):  #Find all adjacent plunge/escape transitions
        
        b_1, fate_1 = scan_results[i]
        b_2, fate_2 = scan_results[i + 1]
        
        if fate_1 != fate_2 and {fate_1, fate_2} == {"plunge", "escape"}:

            transition_brackets.append((b_1, fate_1, b_2, fate_2))
            
    if metric == "kerr":

        if len(transition_brackets) != 1:

            print()
            print("WARNING: Kerr scan is not monotonic.")
            print("Number of plunge/escape transitions =", len(transition_brackets))
            print("Transitions =", transition_brackets)

            return (np.nan, np.nan, b_scan_ray_data, [], [], np.nan, "manual needed")
            
        selected_bracket = transition_brackets[0]

    elif metric == "quasi_kerr":

        if len(transition_brackets) == 0:

            print()
            print("No plunge/escape transition found for "f"ptheta={ptheta0:.4f}")

            return (np.nan, np.nan, b_scan_ray_data, [], [], np.nan, "manual needed")

        selected_bracket = max(transition_brackets, key=lambda x: max(x[0], x[2]))  #Choose the outermost transition (subject to change for specific metric)

        if len(transition_brackets) > 1:

            print()
            print("Multiple quasi-Kerr transitions found:")
            print(transition_brackets)
            print("Using outermost transition:")
            print(selected_bracket)
    else:

        print("Unknown metric:", metric)

        return (np.nan, np.nan, b_scan_ray_data, [], [], np.nan, "unknown metric")

    b_1, fate_1, b_2, fate_2 = selected_bracket   #Assign plunge and escape endpoints
    
    if fate_1 == "plunge":

        b_plunge = b_1
        b_escape = b_2

    else:

        b_plunge = b_2
        b_escape = b_1
        
    print()
    print("Initial b bracket:")
    print("b_plunge =", direction * b_plunge)
    print("b_escape =", direction * b_escape)

    
    # --------------------------------------------------
    # Bisection method to find b_crit
    # --------------------------------------------------

    b_bisection_history = []
    iteration = 0
    
    while abs(b_escape - b_plunge) > bisection_tol:

        b_mid = 0.5 * (b_plunge + b_escape)

        tracks, fate, phi_total = shoot_ray_ivp(b_mid, a, metric=metric, ptheta0=ptheta0, theta_obs=theta_obs, direction=direction, epsilon_qk=epsilon_qk)

        if fate == "plunge":

            b_plunge = b_mid   #Update bracket

        elif fate == "escape":
        
            b_escape = b_mid   #Update bracket 

        else:
    
            break
            
        b_bisection_history.append({     #Save bisection data for each iteration for error analysis
            "iteration": iteration,
            "a": a,
            "theta_obs": theta_obs,
            "direction": direction,
        
            "b": np.nan,
            "ptheta": ptheta0,
            "psi": np.nan,
        
            "b_mid": direction * b_mid,
            "ptheta_mid": np.nan,
            "rho_mid": np.nan,
        
            "fate": fate,
        
            "b_plunge": direction * b_plunge,
            "b_escape": direction * b_escape,
        
            "ptheta_plunge": np.nan,
            "ptheta_escape": np.nan,
        
            "rho_plunge": np.nan,
            "rho_escape": np.nan,
        
            "critical_estimate": direction * 0.5 * (b_plunge + b_escape),
            "bracket_width": abs(b_escape - b_plunge)
        })
    
        iteration += 1

    # --------------------------------------------------
    # Final critical impact parameter
    # --------------------------------------------------

    bcrit_unsigned = 0.5 * (b_plunge + b_escape)
    bcrit_signed = direction * bcrit_unsigned


    # --------------------------------------------------
    # Shoot ONE ray at final b_crit
    # --------------------------------------------------

    critical_tracks, critical_fate, critical_phi_total = shoot_ray_ivp(bcrit_unsigned, a, metric, ptheta0=ptheta0, theta_obs=theta_obs, direction=direction, epsilon_qk=epsilon_qk)

    # --------------------------------------------------
    # Estimate critical radius
    # --------------------------------------------------

    r_track = np.asarray(critical_tracks["r"])
    r_dot_track = np.asarray(critical_tracks["r_dot"])

    ncrit = min(len(r_track), len(r_dot_track))

    if ncrit > 0:

        rcrit_estimate = r_track[:ncrit][np.argmin(np.abs(r_dot_track[:ncrit]))]    #when rdot is closest to zero, find the corresponding radial coordinate 

    elif len(r_track) > 0:

        rcrit_estimate = np.min(r_track)

    else:

        rcrit_estimate = np.nan


    # --------------------------------------------------
    # Save critical trajectory
    # --------------------------------------------------

    b_critical_ray_data = [
    make_ray_record(
        tracks=critical_tracks,
        a=a,
        theta_obs=theta_obs,
        b=bcrit_signed,
        b_unsigned=bcrit_unsigned,
        ptheta=ptheta0,
        direction=direction,
        fate=critical_fate,
        phi_total=critical_phi_total)]

    b_critical_ray_data[0]["rcrit_estimate"] = rcrit_estimate
    b_critical_ray_data[0]["bracket_width"] = abs(b_escape - b_plunge)
    
    b_critical_ray_data[0]["b_plunge_final"] = direction * b_plunge
    b_critical_ray_data[0]["b_escape_final"] = direction * b_escape
    
    b_critical_ray_data[0]["ptheta_plunge_final"] = np.nan
    b_critical_ray_data[0]["ptheta_escape_final"] = np.nan
    
    b_critical_ray_data[0]["rho_plunge_final"] = np.nan
    b_critical_ray_data[0]["rho_escape_final"] = np.nan


    alpha = -bcrit_signed / np.sin(theta_obs)
    beta = ptheta0

    print()
    print("----------------------------------------")
    print("Critical image-plane point found")
    print("bcrit =", bcrit_signed)
    print("ptheta =", ptheta0)
    print("alpha =", alpha)
    print("beta =", beta)
    print("direction =", direction)
    print("rcrit estimate =", rcrit_estimate)
    print("bisection_tol", bisection_tol)
    print("----------------------------------------")


    return (bcrit_unsigned, bcrit_signed, b_scan_ray_data, b_critical_ray_data, b_bisection_history, rcrit_estimate, "ok")

In [123]:
def run_fixed_ptheta_point(ptheta0, b_values, a, metric, theta_obs, direction=1, epsilon_qk=0.0):

    result = find_bcrit_at_ptheta_ivp(
        ptheta0=ptheta0,
        b_values=b_values,
        a=a,
        metric=metric,
        theta_obs=theta_obs,
        direction=direction,
        epsilon_qk=epsilon_qk)

    return ptheta0, direction, result

In [141]:
def find_pthetacrit_at_b_ivp(b_target, ptheta_values, a, metric, theta_obs, direction=1, epsilon_qk=0.0):  #Fix b (~alpha) and find corresponding critical ptheta (beta)

    b_unsigned = abs(b_target)
    b_signed = b_unsigned * direction


    # --------------------------------------------------------
    # Check that b_target and direction agree
    # --------------------------------------------------------

    if a != 0:

        if not np.isclose(b_signed, b_target):

            print("Wrong direction for this b_target")
            print("b_target =", b_target)
            print("direction =", direction)
            print("this direction gives b_signed =", b_signed)

            return (np.nan, np.nan, [], [], [], np.nan, "wrong direction")

    # --------------------------------------------------------
    # Scan ptheta values to find plunge/escape bracket
    # --------------------------------------------------------
    
    ptheta_scan_ray_data = []
    scan_results = []
    
    for ptheta0 in ptheta_values:
    
        tracks, fate, phi_total = shoot_ray_ivp(
            b_unsigned,
            a,
            metric=metric,
            ptheta0=ptheta0,
            theta_obs=theta_obs,
            direction=direction,
            epsilon_qk=epsilon_qk)
    
        alpha = -b_signed / np.sin(theta_obs)
        beta = ptheta0
        
        print(f"b={b_signed:.4f}", f"ptheta={ptheta0:.4f}", f"alpha={alpha:.4f}", f"beta={beta:.4f}", fate)

        ptheta_scan_ray_data.append(
            make_ray_record(
                tracks=tracks,
                a=a,
                theta_obs=theta_obs,
                b=b_signed,
                b_unsigned=b_unsigned,
                ptheta=ptheta0,
                direction=direction,
                fate=fate,
                phi_total=phi_total,
                alpha=alpha,
                beta=beta))
        
        scan_results.append((ptheta0, fate))
    transition_brackets = []

    for i in range(len(scan_results) - 1):  # Find all adjacent plunge/escape transitions
    
        ptheta_1, fate_1 = scan_results[i]
        ptheta_2, fate_2 = scan_results[i + 1]
    
        if fate_1 != fate_2 and {fate_1, fate_2} == {"plunge", "escape"}:
        
            transition_brackets.append((ptheta_1, fate_1, ptheta_2, fate_2))

    if metric == "kerr":

        if len(transition_brackets) != 1:
    
            print("WARNING: Kerr scan is not monotonic.")
            print("Number of plunge/escape transitions =", len(transition_brackets))
            print("Transitions =", transition_brackets)
    
            return (np.nan, b_signed, ptheta_scan_ray_data, [], [], np.nan, "manual needed")   #Requires user to look into this specific case

        selected_bracket = transition_brackets[0]
        
    elif metric == "quasi_kerr":

        if len(transition_brackets) == 0:
    
            print()
            print("No plunge/escape transition found for "f"b={b_signed:.4f}")
    
            return (np.nan, b_signed, ptheta_scan_ray_data, [], [], np.nan, "manual needed")
    
        selected_bracket = max(transition_brackets, key=lambda x: max(x[0], x[2]))   #Choose the outermost transition (subject to change for specific metric)
    
        if len(transition_brackets) > 1:

            print("Multiple quasi-Kerr transitions found:")
            print(transition_brackets)
            print("Using outermost transition:")
            print(selected_bracket)
    else:

        print("Unknown metric:", metric)
    
        return (np.nan, b_signed, ptheta_scan_ray_data, [], [], np.nan, "unknown metric")
                
    ptheta_1, fate_1, ptheta_2, fate_2 = selected_bracket   #Assign plunge and escape endpoints

    if fate_1 == "plunge":    
        ptheta_plunge = ptheta_1
        ptheta_escape = ptheta_2
    
    else:
        ptheta_plunge = ptheta_2
        ptheta_escape = ptheta_1


    print()
    print("Initial ptheta bracket:")
    print("ptheta_plunge =", ptheta_plunge)
    print("ptheta_escape =", ptheta_escape)


    # --------------------------------------------------------
    # Bisection in ptheta
    # --------------------------------------------------------
    ptheta_bisection_history = []
    iteration = 0
    
    while abs(ptheta_escape - ptheta_plunge) > bisection_tol:
    
        ptheta_mid = 0.5 * (ptheta_plunge + ptheta_escape)
    
        tracks, fate, phi_total = shoot_ray_ivp(
            b_unsigned,
            a,
            metric=metric,
            ptheta0=ptheta_mid,
            theta_obs=theta_obs,
            direction=direction,
            epsilon_qk=epsilon_qk)
    
        print(
            "ptheta_plunge:", ptheta_plunge,
            "ptheta_escape:", ptheta_escape,
            "ptheta_mid:", ptheta_mid,
            "fate:", fate)
        
        if fate == "plunge":
    
            ptheta_plunge = ptheta_mid
    
            
        elif fate == "escape":

            ptheta_escape = ptheta_mid
    
        else:
    
            break
            
        ptheta_bisection_history.append({    #Save bisection data for each iteration for error analysis
            "iteration": iteration,
            "a": a,
            "theta_obs": theta_obs,
            "direction": direction,
        
            "b": b_signed,
            "ptheta": np.nan,
            "psi": np.nan,
        
            "b_mid": np.nan,
            "ptheta_mid": ptheta_mid,
            "rho_mid": np.nan,
        
            "fate": fate,
        
            "b_plunge": np.nan,
            "b_escape": np.nan,
        
            "ptheta_plunge": ptheta_plunge,
            "ptheta_escape": ptheta_escape,
        
            "rho_plunge": np.nan,
            "rho_escape": np.nan,
        
            "critical_estimate": 0.5 * (ptheta_plunge + ptheta_escape),
            "bracket_width": abs(ptheta_escape - ptheta_plunge)
        })
    
        iteration += 1

    # --------------------------------------------------------
    # Final critical ptheta
    # --------------------------------------------------------

    ptheta_crit = 0.5 * (ptheta_plunge + ptheta_escape)


    # --------------------------------------------------------
    # Shoot final critical ray
    # --------------------------------------------------------

    critical_tracks, critical_fate, critical_phi_total = shoot_ray_ivp(b_unsigned, a, metric=metric, ptheta0=ptheta_crit, theta_obs=theta_obs, direction=direction, epsilon_qk=epsilon_qk)


    # --------------------------------------------------------
    # Estimate critical radius
    # --------------------------------------------------------

    r_track = np.asarray(critical_tracks["r"])

    r_dot_track = np.asarray(critical_tracks["r_dot"])

    ncrit = min(len(r_track), len(r_dot_track))

    if ncrit > 0:

        rcrit_estimate = r_track[:ncrit][np.argmin(np.abs(r_dot_track[:ncrit]))]

    elif len(r_track) > 0:

        rcrit_estimate = np.min(r_track)

    else:

        rcrit_estimate = np.nan


    # --------------------------------------------------------
    # Save final critical trajectory
    # --------------------------------------------------------

    ptheta_critical_ray_data = [
    make_ray_record(
        tracks=critical_tracks,
        a=a,
        theta_obs=theta_obs,
        b=b_signed,
        b_unsigned=b_unsigned,
        ptheta=ptheta_crit,
        direction=direction,
        fate=critical_fate,
        phi_total=critical_phi_total)]

    ptheta_critical_ray_data[0]["rcrit_estimate"] = rcrit_estimate
    ptheta_critical_ray_data[0]["bracket_width"] = abs(ptheta_escape - ptheta_plunge)
    
    ptheta_critical_ray_data[0]["b_plunge_final"] = np.nan
    ptheta_critical_ray_data[0]["b_escape_final"] = np.nan
    
    ptheta_critical_ray_data[0]["ptheta_plunge_final"] = ptheta_plunge
    ptheta_critical_ray_data[0]["ptheta_escape_final"] = ptheta_escape
    
    ptheta_critical_ray_data[0]["rho_plunge_final"] = np.nan
    ptheta_critical_ray_data[0]["rho_escape_final"] = np.nan


    # --------------------------------------------------------
    # Final diagnostic
    # --------------------------------------------------------

    alpha = -b_signed / np.sin(theta_obs)
    beta = ptheta_crit

    print()
    print("----------------------------------------")
    print("Critical image-plane point found")
    print("b =", b_signed)
    print("ptheta_crit =", ptheta_crit)
    print("alpha =", alpha)
    print("beta =", beta)
    print("direction =", direction)
    print("rcrit estimate =", rcrit_estimate)
    print("bisection_tol", bisection_tol)
    print("----------------------------------------")


    return (ptheta_crit, b_signed, ptheta_scan_ray_data, ptheta_critical_ray_data, ptheta_bisection_history, rcrit_estimate, "ok")

In [143]:
def run_fixed_b_point(b_target, ptheta_values, a, metric, theta_obs, direction=1, epsilon_qk=0.0):

    result = find_pthetacrit_at_b_ivp(
        b_target=b_target,
        ptheta_values=ptheta_values,
        a=a,
        metric=metric,
        theta_obs=theta_obs,
        direction=direction,
        epsilon_qk=epsilon_qk)

    return b_target, direction, result

In [129]:
def find_rhocrit_at_psi_ivp(psi, rho_values, a, metric, theta_obs, epsilon_qk=0.0): #Find critical image-plane radius at fixed angle psi

    rho_scan_ray_data = []
    scan_results = []

    # --------------------------------------------------------
    # Scan rho values to find plunge/escape bracket
    # --------------------------------------------------------

    for rho in rho_values:

        alpha = rho * np.cos(psi)
        beta = rho * np.sin(psi)

        b_signed = -alpha * np.sin(theta_obs)
        b_unsigned = abs(b_signed)

        ptheta0 = beta

        direction = 1 if b_signed >= 0 else -1

        tracks, fate, phi_total = shoot_ray_ivp(b_unsigned, a, metric=metric, ptheta0=ptheta0, theta_obs=theta_obs, direction=direction, epsilon_qk=epsilon_qk)

        print(
            f"psi={np.rad2deg(psi):.1f} deg",
            f"rho={rho:.4f}",
            f"alpha={alpha:.4f}",
            f"beta={beta:.4f}",
            fate)

        rho_scan_ray_data.append(make_ray_record(
            tracks=tracks,
            a=a,
            theta_obs=theta_obs,
            b=b_signed,
            b_unsigned=b_unsigned,
            ptheta=ptheta0,
            direction=direction,
            fate=fate,
            phi_total=phi_total,
            alpha=alpha,
            beta=beta,
            psi=psi,
            rho=rho))
        scan_results.append((rho, fate))
        
    transition_brackets = []

    for i in range(len(scan_results) - 1):  #Find all adjacent plunge/escape transitions
    
        rho_1, fate_1 = scan_results[i]
        rho_2, fate_2 = scan_results[i + 1]
    
        if fate_1 != fate_2 and {fate_1, fate_2} == {"plunge", "escape"}:
    
            transition_brackets.append((rho_1, fate_1, rho_2, fate_2))

    if metric == "kerr":

        if len(transition_brackets) != 1:
    
            print()
            print("WARNING: Kerr scan is not monotonic.")
            print("Number of plunge/escape transitions =", len(transition_brackets))
            print("Transitions =", transition_brackets)
            return (np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, rho_scan_ray_data, [], [], np.nan, "manual needed")
        selected_bracket = transition_brackets[0]
        
    elif metric == "quasi_kerr":

        if len(transition_brackets) == 0:

            print()
            print("No plunge/escape transition found for "f"psi={np.rad2deg(psi):.1f} deg")
            return (np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, rho_scan_ray_data, [], [], np.nan, "manual needed")
            
        selected_bracket = max(transition_brackets, key=lambda x: max(x[0], x[2]))  #Choose the outermost transition (subject to change for specific metric)
        
        if len(transition_brackets) > 1:

            print()
            print("Multiple quasi-Kerr transitions found:")
            print(transition_brackets)
            print("Using outermost transition:")
            print(selected_bracket)
    else:

        print("Unknown metric:", metric)
        return (np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, rho_scan_ray_data, [], [], np.nan, "unknown metric")


    rho_1, fate_1, rho_2, fate_2 = selected_bracket   #Assign plunge and escape endpoints

    if fate_1 == "plunge":

        rho_plunge = rho_1
        rho_escape = rho_2

    else:

        rho_plunge = rho_2
        rho_escape = rho_1
        
    print()
    print("Initial rho bracket:")
    print("rho_plunge =", rho_plunge)
    print("rho_escape =", rho_escape)


    # --------------------------------------------------------
    # Bisection in rho
    # --------------------------------------------------------
    
    rho_bisection_history = []
    iteration = 0
    
    
    while abs(rho_escape - rho_plunge) > bisection_tol:
    
        rho_mid = 0.5 * (rho_plunge + rho_escape)
        bracket_width = abs(rho_escape - rho_plunge)
    
        alpha_mid = rho_mid * np.cos(psi)
        beta_mid = rho_mid * np.sin(psi)
    
        b_signed_mid = -alpha_mid * np.sin(theta_obs)
        b_unsigned_mid = abs(b_signed_mid)
    
        ptheta_mid = beta_mid
    
        direction_mid = 1 if b_signed_mid >= 0 else -1
    
        tracks, fate, phi_total = shoot_ray_ivp(b_unsigned_mid, a, metric=metric, ptheta0=ptheta_mid, theta_obs=theta_obs, direction=direction_mid, epsilon_qk=epsilon_qk)

        # ----------------------------------------------------
        # Update plunge side
        # ----------------------------------------------------
    
        if fate == "plunge":
    
            rho_plunge = rho_mid

        # ----------------------------------------------------
        # Update escape side
        # ----------------------------------------------------
    
        elif fate == "escape":
    
            rho_escape = rho_mid

        else:
    
            break
            
        rho_bisection_history.append({   #Save bisection data for each iteration for error analysis
            "iteration": iteration,
            "a": a,
            "theta_obs": theta_obs,
            "direction": np.nan,
        
            "b": np.nan,
            "ptheta": np.nan,
            "psi": psi,
        
            "b_mid": np.nan,
            "ptheta_mid": np.nan,
            "rho_mid": rho_mid,
        
            "fate": fate,
        
            "b_plunge": np.nan,
            "b_escape": np.nan,
        
            "ptheta_plunge": np.nan,
            "ptheta_escape": np.nan,
        
            "rho_plunge": rho_plunge,
            "rho_escape": rho_escape,
        
            "critical_estimate": 0.5 * (rho_plunge + rho_escape),
            "bracket_width": abs(rho_escape - rho_plunge)
        })
    
        iteration += 1
        
    # --------------------------------------------------------
    # Final critical image-plane point
    # --------------------------------------------------------

    rho_crit = 0.5 * (rho_plunge + rho_escape)

    alpha_crit = rho_crit * np.cos(psi)
    beta_crit = rho_crit * np.sin(psi)

    bcrit_signed = (-alpha_crit * np.sin(theta_obs))

    bcrit_unsigned = abs(bcrit_signed)

    ptheta_crit = beta_crit

    direction_crit = (1 if bcrit_signed >= 0 else -1)

    # --------------------------------------------------------
    # Shoot final critical ray
    # --------------------------------------------------------

    critical_tracks, critical_fate, critical_phi_total = shoot_ray_ivp(bcrit_unsigned, a, metric=metric, ptheta0=ptheta_crit, theta_obs=theta_obs, direction=direction_crit, epsilon_qk=epsilon_qk)

    # --------------------------------------------------------
    # Estimate critical radius
    # --------------------------------------------------------

    r_track = np.asarray(critical_tracks["r"])

    r_dot_track = np.asarray(critical_tracks["r_dot"])

    ncrit = min(len(r_track), len(r_dot_track))

    if ncrit > 0:

        rcrit_estimate = r_track[:ncrit][np.argmin(np.abs(r_dot_track[:ncrit]))]

    elif len(r_track) > 0:

        rcrit_estimate = np.min(r_track)

    else:

        rcrit_estimate = np.nan


    # --------------------------------------------------------
    # Save final critical trajectory
    # --------------------------------------------------------

    rho_critical_ray_data = [
    make_ray_record(
        tracks=critical_tracks,
        a=a,
        theta_obs=theta_obs,
        b=bcrit_signed,
        b_unsigned=bcrit_unsigned,
        ptheta=ptheta_crit,
        direction=direction_crit,
        fate=critical_fate,
        phi_total=critical_phi_total,
        alpha=alpha_crit,
        beta=beta_crit,
        psi=psi,
        rho=rho_crit)]

    rho_critical_ray_data[0]["rcrit_estimate"] = rcrit_estimate
    rho_critical_ray_data[0]["bracket_width"] = abs(rho_escape - rho_plunge)
    
    rho_critical_ray_data[0]["b_plunge_final"] = np.nan
    rho_critical_ray_data[0]["b_escape_final"] = np.nan
    
    rho_critical_ray_data[0]["ptheta_plunge_final"] = np.nan
    rho_critical_ray_data[0]["ptheta_escape_final"] = np.nan
    
    rho_critical_ray_data[0]["rho_plunge_final"] = rho_plunge
    rho_critical_ray_data[0]["rho_escape_final"] = rho_escape


    print()
    print("----------------------------------------")
    print("Critical image-plane point found")
    print("psi =", np.rad2deg(psi), "deg")
    print("rho_crit =", rho_crit)
    print("alpha =", alpha_crit)
    print("beta =", beta_crit)
    print("bcrit =", bcrit_signed)
    print("ptheta =", ptheta_crit)
    print("rcrit estimate =", rcrit_estimate)
    print("bisection_tol", bisection_tol)
    print("----------------------------------------")


    return (
        rho_crit,
        alpha_crit,
        beta_crit,
        bcrit_signed,
        ptheta_crit,
        direction_crit,
        rho_scan_ray_data, 
        rho_critical_ray_data,
        rho_bisection_history,
        rcrit_estimate,
        "ok")

In [131]:
def run_image_plane_point(psi, rho_values, a, metric, theta_obs, epsilon_qk=0.0):

    result = find_rhocrit_at_psi_ivp(
        psi=psi,
        rho_values=rho_values,
        a=a,
        metric=metric,
        theta_obs=theta_obs,
        epsilon_qk=epsilon_qk)

    return psi, result

In [133]:
def rays_to_dataframe(rays):   #Preparing csv
    
    rows = []

    for ray in rays:
        
        if ray is None:
            continue

        # Make sure all tracked arrays have the same length
        n = min(len(ray[k]) for k in TRACK_KEYS)

        if n == 0:
            continue

        # Trajectory data
        data = {k: ray[k][:n] for k in TRACK_KEYS}

        # Information identifying the ray
        meta = {
        "b": ray["b"],
        "b_unsigned": ray["b_unsigned"],
        "direction": ray["direction"],
        "a": ray["a"],
        "ptheta": ray["ptheta"],
        "fate": ray.get("fate", ""),
        "phi_total": ray.get("phi_total", np.nan),
    
        # Image-plane information
        "alpha": ray.get("alpha", np.nan),
        "beta": ray.get("beta", np.nan),
        "psi": ray.get("psi", np.nan),
        "psi_deg": ray.get("psi_deg", np.nan),
        "rho": ray.get("rho", np.nan),
    
        # Critical-ray information
        "rcrit_estimate": ray.get("rcrit_estimate", np.nan),
    
        # Bisection information
        "bracket_side": ray.get("bracket_side", ""),
        "bracket_width": ray.get("bracket_width", np.nan),
    
        "b_plunge_final": ray.get("b_plunge_final", np.nan),
        "b_escape_final": ray.get("b_escape_final", np.nan),
    
        "ptheta_plunge_final": ray.get("ptheta_plunge_final", np.nan),
        "ptheta_escape_final": ray.get("ptheta_escape_final", np.nan),
    
        "rho_plunge_final": ray.get("rho_plunge_final", np.nan),
        "rho_escape_final": ray.get("rho_escape_final", np.nan)}

        # Repeat metadata for every trajectory step
        for key, value in meta.items():
            data[key] = np.full(n, value)

        rows.append(pd.DataFrame(data))

    if rows:
        return pd.concat(rows, ignore_index=True)
    
    return pd.DataFrame()


# ============================================================
# Automatic filename formatting
# ============================================================

def filename_tag(metric, a, theta_obs_deg, ptheta0=None, epsilon_qk=None):

    tag = f"{metric}_a_{a:.2f}_inc_{theta_obs_deg:.1f}"
    
    # Only add epsilon for quasi-Kerr data
    if metric == 'quasi_kerr':
        tag += f"_eps_{epsilon_qk:+.2f}"
        
    # Only add ptheta for individual trajectory files
    if ptheta0 is not None:
        tag += f"_ptheta_{ptheta0:+.2f}"
    
    return tag

In [135]:
#Change these parameters for a specific data set/metric

a = 0.98   #spin (0 -> 1)

theta_obs_deg = 90  #inclination angle (0 -> 90)

theta_obs = np.deg2rad(theta_obs_deg)

metric = 'kerr'  #or "quasi_kerr"
epsilon_qk = 0.00  #deviation parameter for quasi-kerr, 0 = kerr

# ------------------------------------------------------------
# Sampling method
# ------------------------------------------------------------

SAMPLING_METHOD = 1

# 0 = fixed ptheta -> find bcrit
# 1 = fixed b      -> find ptheta_crit
# 2 = image-plane angle -> find rho_crit


# ------------------------------------------------------------
# Sampling values
# ------------------------------------------------------------

# Used only for SAMPLING_METHOD = 0

PTHETA_VALUES = np.linspace(-6, 6, 20, endpoint=True)


# Used only for SAMPLING_METHOD = 1

B_TARGET_VALUES = np.linspace(-6, 6, 20, endpoint=True)

# Used only for SAMPLING_METHOD = 2

PSI_VALUES = np.linspace(0, 2 * np.pi, 40, endpoint=False)


if abs(a) >= 0.8:    # Offset high-spin sampling to avoid error at endpoints, psi = 0 and 180 deg
    PSI_VALUES += np.pi / 40
    
# ------------------------------------------------------------
# Bracketing / bisection ranges
# ------------------------------------------------------------

b_values = np.linspace(0, 10, 20)

ptheta_values_upper = np.linspace(0, 10, 20)
ptheta_values_lower = np.linspace(0, -10, 20)

rho_values = np.linspace(0, 10, 20)

# ------------------------------------------------------------
# Saving
# ------------------------------------------------------------

SAVE_CRITICAL_CURVE = True  #save critical curve points
SAVE_REGULAR_DATA = False    #save trajectory info for initial 20 rays 
SAVE_CRITICAL_DATA = False   #save final critical trajectory info 
SAVE_BISECTION_DATA = False   #save bisection bracketing iteration info 

# ------------------------------------------------------------
# Run summary
# ------------------------------------------------------------

print("=" * 50)
print("RUN SETTINGS")
print("=" * 50)

print("Metric:", metric)
print("Spin:", a)
print("Inclination:", theta_obs_deg)

if SAMPLING_METHOD == 0:
    print("Sampling: fixed ptheta, find bcrit")
    print("ptheta values:", PTHETA_VALUES)

elif SAMPLING_METHOD == 1:
    print("Sampling: fixed b, find ptheta_crit")
    print("b target values:", B_TARGET_VALUES)

elif SAMPLING_METHOD == 2:
    print("Sampling: image-plane angle, find rho_crit")
    print("Number of angles:", len(PSI_VALUES))

print()
print("Save critical curve:", SAVE_CRITICAL_CURVE)
print("Save regular data:", SAVE_REGULAR_DATA)
print("Save critical data:", SAVE_CRITICAL_DATA)
print("Save bisection data:", SAVE_BISECTION_DATA)

print("=" * 50)

RUN SETTINGS
Metric: kerr
Spin: 0.98
Inclination: 90
Sampling: fixed b, find ptheta_crit
b target values: [-6.         -5.36842105 -4.73684211 -4.10526316 -3.47368421 -2.84210526
 -2.21052632 -1.57894737 -0.94736842 -0.31578947  0.31578947  0.94736842
  1.57894737  2.21052632  2.84210526  3.47368421  4.10526316  4.73684211
  5.36842105  6.        ]

Save critical curve: True
Save regular data: False
Save critical data: False
Save bisection data: False


In [145]:
import time
# ============================================================
# Generate critical curve data (run this cell)
# ============================================================

critical_curve_results = []
regular_ray_results = []
critical_ray_results = []

b_bisection_history_results = []
ptheta_bisection_history_results = []
rho_bisection_history_results = []


# ============================================================
# METHOD 0: Fixed ptheta, find bcrit
# ============================================================

if SAMPLING_METHOD == 0:
    
    # --------------------------------------------------------
    # Build independent jobs:
    # every ptheta value × both b directions
    # --------------------------------------------------------
    tasks = [(ptheta0, direction) for ptheta0 in PTHETA_VALUES for direction in [1, -1]]

    # --------------------------------------------------------
    # Run jobs in parallel
    # --------------------------------------------------------

    results = Parallel(
        n_jobs=4,
        backend="loky",
        verbose=10
    )(
        delayed(run_fixed_ptheta_point)(
            ptheta0=ptheta0,
            direction=direction,
            a=a,
            metric=metric,
            theta_obs=theta_obs,
            epsilon_qk=epsilon_qk,
            b_values=b_values
        )
        for ptheta0, direction in tasks
    )

    # --------------------------------------------------------
    # Collect results
    # --------------------------------------------------------

    for ptheta0, direction, result in results:

        (bcrit_unsigned, bcrit_signed, b_scan_ray_data, b_critical_ray_data, b_bisection_history, rcrit_estimate, status) = result
    
        regular_ray_results.extend(b_scan_ray_data)
    
        critical_ray_results.extend(b_critical_ray_data)
    
        b_bisection_history_results.extend(b_bisection_history)
    
        critical_curve_results.append({
            "sampling_method": "fixed_ptheta",
            "a": a,
            "theta_obs_deg": theta_obs_deg,
            "ptheta": ptheta0,
            "direction": direction,
            "bcrit": bcrit_signed,
            "bcrit_unsigned": bcrit_unsigned,
            "rcrit_estimate": rcrit_estimate,
            "status": status})


        print()
        print("=" * 50)
        print(f"a = {a}")
        print(f"inclination = {theta_obs_deg} deg")
        print(f"p_theta = {ptheta0}")
        print(f"direction = {direction}")
        print(f"bcrit = {bcrit_signed}")
        print(f"rcrit = {rcrit_estimate}")
        print("=" * 50)

# ============================================================
# METHOD 1: Fixed b, find ptheta_crit
# ============================================================

elif SAMPLING_METHOD == 1:

    # --------------------------------------------------------
    # Run each b_target independently in parallel
    # --------------------------------------------------------
    tasks = []
    for b_target in B_TARGET_VALUES:

        direction = 1 if b_target >= 0 else -1
    
        tasks.append((b_target, direction, ptheta_values_upper))
        tasks.append((b_target, direction, ptheta_values_lower))

    results = Parallel(
    n_jobs=4,
    backend="loky",
    verbose=10)(
    delayed(run_fixed_b_point)(
        b_target=b_target,
        a=a,
        metric=metric,
        theta_obs=theta_obs,
        epsilon_qk=epsilon_qk,
        ptheta_values=ptheta_scan,
        direction=direction)
    for b_target, direction, ptheta_scan in tasks)

    # --------------------------------------------------------
    # Collect results
    # --------------------------------------------------------

    for b_target, direction, result in results:

        (ptheta_crit, bcrit, ptheta_scan_ray_data, ptheta_critical_ray_data, ptheta_bisection_history, rcrit_estimate, status) = result


        # -----------------------------------------
        # Store trajectory data
        # -----------------------------------------

        regular_ray_results.extend(ptheta_scan_ray_data)

        critical_ray_results.extend(ptheta_critical_ray_data)

        ptheta_bisection_history_results.extend(ptheta_bisection_history)


        # -----------------------------------------
        # Store critical curve point
        # -----------------------------------------

        critical_curve_results.append({
            "sampling_method": "fixed_b",
            "a": a,
            "theta_obs_deg": theta_obs_deg,
            "ptheta": ptheta_crit,
            "direction": direction,
            "bcrit": bcrit,
            "bcrit_unsigned": abs(bcrit),
            "rcrit_estimate": rcrit_estimate,
            "status": status})


        print()
        print("=" * 50)
        print(f"a = {a}")
        print(f"inclination = {theta_obs_deg} deg")
        print(f"b_target = {b_target}")
        print(f"direction = {direction}")
        print(f"ptheta_crit = {ptheta_crit}")
        print(f"bcrit = {bcrit}")
        print(f"rcrit = {rcrit_estimate}")
        print("=" * 50)
        


# ============================================================
# METHOD 2: Fix Image-plane angle, find rho_crit, calculate corresponding alpha, beta
# =============================================================

elif SAMPLING_METHOD == 2:

    results = Parallel(
        n_jobs=4,
        backend="loky",
        verbose=10
    )(
        delayed(run_image_plane_point)(
            psi=psi,
            rho_values=rho_values,
            a=a,
            metric=metric,
            theta_obs=theta_obs,
            epsilon_qk=epsilon_qk
        )
        for psi in PSI_VALUES)


    # --------------------------------------------------------
    # Collect completed results
    # --------------------------------------------------------

    for psi, result in results:

        (
            rho_crit,
            alpha_crit,
            beta_crit,
            bcrit,
            ptheta_crit,
            direction,
            rho_scan_ray_data,
            rho_critical_ray_data,
            rho_bisection_history,
            rcrit_estimate,
            status
        ) = result


        regular_ray_results.extend(rho_scan_ray_data)

        critical_ray_results.extend(rho_critical_ray_data)

        rho_bisection_history_results.extend(rho_bisection_history)


        critical_curve_results.append({
            "sampling_method": "fixed_psi",

            "a": a,
            "theta_obs_deg": theta_obs_deg,

            "psi": psi,
            "psi_deg": np.rad2deg(psi),

            "rho_crit": rho_crit,

            "alpha": alpha_crit,
            "beta": beta_crit,

            "ptheta": ptheta_crit,
            "direction": direction,

            "bcrit": bcrit,
            "bcrit_unsigned": abs(bcrit),

            "rcrit_estimate": rcrit_estimate,
            "status": status})


        print()
        print("=" * 50)
        print(f"a = {a}")
        print(f"inclination = {theta_obs_deg} deg")
        print(f"psi = {np.rad2deg(psi):.2f} deg")
        print(f"rho_crit = {rho_crit}")
        print(f"alpha = {alpha_crit}")
        print(f"beta = {beta_crit}")
        print(f"bcrit = {bcrit}")
        print(f"ptheta = {ptheta_crit}")
        print(f"rcrit = {rcrit_estimate}")
        print("=" * 50)

else:

    raise ValueError("SAMPLING_METHOD must be 0, 1, or 2.")

[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   5 tasks      | elapsed:  2.3min
[Parallel(n_jobs=4)]: Done  10 tasks      | elapsed:  4.0min
[Parallel(n_jobs=4)]: Done  17 tasks      | elapsed:  8.4min
[Parallel(n_jobs=4)]: Done  24 tasks      | elapsed: 10.5min
[Parallel(n_jobs=4)]: Done  33 tasks      | elapsed: 11.2min
[Parallel(n_jobs=4)]: Done  38 out of  40 | elapsed: 11.3min remaining:   35.5s



a = 0.98
inclination = 90 deg
b_target = -6.0
direction = -1
ptheta_crit = 3.0549947172403336
bcrit = -6.0
rcrit = 3.8192452262566206

a = 0.98
inclination = 90 deg
b_target = -6.0
direction = -1
ptheta_crit = -3.0549947172403336
bcrit = -6.0
rcrit = 3.8193055531272107

a = 0.98
inclination = 90 deg
b_target = -5.368421052631579
direction = -1
ptheta_crit = 3.791990237016428
bcrit = -5.368421052631579
rcrit = 3.7075629833269375

a = 0.98
inclination = 90 deg
b_target = -5.368421052631579
direction = -1
ptheta_crit = -3.791990213488279
bcrit = -5.368421052631579
rcrit = 3.7073817607159607

a = 0.98
inclination = 90 deg
b_target = -4.7368421052631575
direction = -1
ptheta_crit = 4.311494572382222
bcrit = -4.7368421052631575
rcrit = 3.5909458040578692

a = 0.98
inclination = 90 deg
b_target = -4.7368421052631575
direction = -1
ptheta_crit = -4.311494572382222
bcrit = -4.7368421052631575
rcrit = 3.590803549540773

a = 0.98
inclination = 90 deg
b_target = -4.105263157894737
direction = -1


[Parallel(n_jobs=4)]: Done  40 out of  40 | elapsed: 11.3min finished


In [147]:
# ------------------------------------------------------------
# Sampling method label for filenames
# ------------------------------------------------------------

if SAMPLING_METHOD == 0:
    sampling_tag = "fixed_ptheta"

elif SAMPLING_METHOD == 1:
    sampling_tag = "fixed_b"

elif SAMPLING_METHOD == 2:
    sampling_tag = "fixed_psi"

else:
    raise ValueError("SAMPLING_METHOD must be 0, 1, or 2.")

# ------------------------------------------------------------
# Save regular trajectory data
# ------------------------------------------------------------
    
if SAVE_REGULAR_DATA:

    regular_df = rays_to_dataframe(regular_ray_results)

    if metric == 'kerr':
        tag = filename_tag(metric="kerr", a=a, theta_obs_deg=theta_obs_deg)
        regular_file = (kerr_trajectories_data_dir / f"Regular_Data_{sampling_tag}_{tag}.csv")
        
    elif metric == 'quasi_kerr':
        tag = filename_tag(metric="quasi_kerr", a=a, theta_obs_deg=theta_obs_deg, epsilon_qk = epsilon_qk)
        regular_file = (quasi_kerr_trajectories_data_dir / f"Regular_Data_{sampling_tag}_{tag}.csv")

        
    else:
        raise ValueError("metric must be 'kerr' or 'quasi_kerr'")

    
    regular_df.to_csv(regular_file, index=False)
    print("Saved:", regular_file)

# ------------------------------------------------------------
# Save critical trajectory data
# ------------------------------------------------------------

if SAVE_CRITICAL_DATA:

    critical_df = rays_to_dataframe(critical_ray_results)

    if metric == 'kerr':
        tag = filename_tag(metric="kerr", a=a, theta_obs_deg=theta_obs_deg)

        critical_file = (kerr_critical_trajectories_data_dir / f"Critical_Data_{sampling_tag}_{tag}.csv")

    elif metric == 'quasi_kerr':
        tag = filename_tag(metric="quasi_kerr", a=a, theta_obs_deg=theta_obs_deg, epsilon_qk = epsilon_qk)

        critical_file = (quasi_kerr_critical_trajectories_data_dir / f"Critical_Data_{sampling_tag}_{tag}.csv")
    else:
        raise ValueError("metric must be 'kerr' or 'quasi_kerr'")

    critical_df.to_csv(critical_file, index=False)

    print("Saved:", critical_file)

# ------------------------------------------------------------
# Save / update critical curve data
# ------------------------------------------------------------

if SAVE_CRITICAL_CURVE:

    new_df = pd.DataFrame(critical_curve_results)

    if metric == 'kerr':
        curve_tag = filename_tag(metric="kerr", a=a, theta_obs_deg=theta_obs_deg)
        curve_file = (kerr_critical_curves_data_dir / f"Critical_Curve_{sampling_tag}_{curve_tag}.csv")
    
    elif metric == 'quasi_kerr':
        curve_tag = filename_tag(metric="quasi_kerr", a=a, theta_obs_deg=theta_obs_deg, epsilon_qk = epsilon_qk)
        curve_file = (quasi_kerr_critical_curves_data_dir / f"Critical_Curve_{sampling_tag}_{curve_tag}.csv")
        #curve_file = (qk_testing_data_dir / f"Critical_Curve_QuasiKerr_a_{a:.2f}_inc_{theta_obs_deg:.1f}_eps_{epsilon_qk:+.2f}.csv")
    else:
        raise ValueError("metric must be 'kerr' or 'quasi_kerr'")

    critical_curve_df = new_df   # Create critical curve dataframe

    # --------------------------------------------------------
    # Sort critical-curve points
    # --------------------------------------------------------

    if SAMPLING_METHOD == 0:
        critical_curve_df = critical_curve_df.sort_values(by=["ptheta", "direction"])

    elif SAMPLING_METHOD == 1:
        critical_curve_df = critical_curve_df.sort_values(by=["bcrit"])

    elif SAMPLING_METHOD == 2:
        critical_curve_df = critical_curve_df.sort_values(by=["psi"])

    critical_curve_df = critical_curve_df.reset_index(drop=True)

    critical_curve_df.to_csv(curve_file, index=False)


    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    print()
    print("Saved critical curve:")
    print(curve_file)
    print("Total critical curve points:", len(critical_curve_df))

# ------------------------------------------------------------
# Save bisection trajectory data
# ------------------------------------------------------------

if SAVE_BISECTION_DATA:

    if SAMPLING_METHOD == 0:

        bisection_results = b_bisection_history_results
        bisection_label = "B_Bisection_Data"

    elif SAMPLING_METHOD == 1:

        bisection_results = ptheta_bisection_history_results
        bisection_label = "PTheta_Bisection_Data"

    elif SAMPLING_METHOD == 2:

        bisection_results = rho_bisection_history_results
        bisection_label = "Rho_Bisection_Data"

    else:

        raise ValueError("SAMPLING_METHOD must be 0, 1, or 2.")


    bisection_df = pd.DataFrame(bisection_results)

    if metric == "kerr":

        tag = filename_tag(metric="kerr", a=a, theta_obs_deg=theta_obs_deg)

        bisection_file = (kerr_trajectories_data_dir / f"{bisection_label}_{tag}.csv")

    elif metric == "quasi_kerr":

        tag = filename_tag(metric="quasi_kerr", a=a, theta_obs_deg=theta_obs_deg, epsilon_qk=epsilon_qk)

        bisection_file = (quasi_kerr_trajectories_data_dir / f"{bisection_label}_{tag}.csv")

    else:

        raise ValueError("metric must be 'kerr' or 'quasi_kerr'")

    bisection_df.to_csv(bisection_file, index=False)

    print("Saved:", bisection_file)


Saved critical curve:
E:\PAPER\data\kerr\critical_curves\Critical_Curve_fixed_b_kerr_a_0.98_inc_90.0.csv
Total critical curve points: 40
